<a href="https://colab.research.google.com/github/sabharwal-monish/LLM/blob/main/sft_financial_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "deeplake<4"

In [ ]:
import torch
import torch.nn as nn
import deeplake
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset

## **STEP 1: LOAD & FORMAT DATA**

In [ ]:
print(">>> STEP 1: Streaming FinGPT Data...")
ds = deeplake.load('hub://genai360/FingGPT-sentiment-train-set', read_only=True)
ds_valid = deeplake.load('hub://genai360/FingGPT-sentiment-valid-set', read_only=True)

def prepare_sample_text(example):
  return f'{example['instruction'].text()}\n\n{example['input'].text()}\n\n{example['output'].text()}'

def create_standard_dataset(deep_lake_ds):
  data_list = []
  for item in deep_lake_ds:
    data_list.append({'text': prepare_sample_text(item)})

  return Dataset.from_list(data_list)

print("Converting to Hugging Face format...")
hf_train_dataset = create_standard_dataset(ds)
hf_eval_dataset = create_standard_dataset(ds_valid)
print(f"Data Loaded: {len(hf_train_dataset)} samples.")



# **STEP 2: TOKENIZER & PROCESSING**

In [ ]:

print(">>> STEP 2: Tokenizing...")
model_id = "facebook/opt-1.3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    # Format text for Causal Language Modeling (Labels = Input_Ids)
    outputs = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    outputs["labels"] = outputs["input_ids"].copy() # Copy input to labels for training
    return outputs

# FIX: We add `remove_columns=["text"]` to drop the raw string column
# This prevents the Trainer from choking on the string data
tokenized_train = hf_train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"] # <--- THE CRITICAL FIX
)

tokenized_eval = hf_eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"] # <--- THE CRITICAL FIX
)

# **STEP 3: MODEL SETUP**

In [ ]:
print(">>> STEP 3: Loading Model...")

# Check Hardware (Use FP16 for T4 GPU, FP32 for CPU)
# We avoid BF16 because T4 GPUs don't support it well
device_type = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Using Precision: {device_type}")

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=device_type)

# Stability Hacks (Freeze & Cast)
for param in model.parameters():
    param.requires_grad = False
    if param.ndim == 1:
        param.data = param.data.to(torch.float32)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

class CastOutputToFloat(nn.Sequential):
    def forward(self, x): return super().forward(x).to(torch.float32)
model.lm_head = CastOutputToFloat(model.lm_head)

# **STEP 4: INJECT LoRA**

In [ ]:
print(">>> STEP 4: Injecting LoRA Adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# **STEP 5: TRAIN**

In [ ]:
print(">>> STEP 5: Initializing Trainer...")

training_args = TrainingArguments(
    output_dir="./FinGPT-Opt-LoRA",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    max_steps=50,
    logging_steps=10,
    fp16=True if torch.cuda.is_available() else False,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print("Starting Training...")
trainer.train()

# **STEP 6: MERGE & TEST**

In [ ]:

print(">>> STEP 6: Merging and Testing...")

# 1. Merge Weights
if hasattr(model, "merge_and_unload"):
    model = model.merge_and_unload()

# 2. FIX: Switch to Evaluation Mode and standardized dtype
model.eval()
# If on GPU, convert everything to FP16. If CPU, FP32.
target_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model.to(dtype=target_dtype, device=device)

# 3. Test
prompt = """What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}

Content: Apple stock plummets 10% after earnings report misses expectations.

Sentiment:"""

inputs = tokenizer(prompt, return_tensors="pt")

# Ensure inputs match the model's device and dtype
inputs = {k: v.to(device) for k, v in inputs.items()}

# Run Generation
outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False)
print("\n--- Model Prediction ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("------------------------")